# 足球SDK完整能力验证

按[完整操作与记录](足球测试_第4轮_操作与记录.md)在同一次现场测试中完成9步，最后统一交回记录。覆盖创建与全部四个公开方法：start、stop、update_command、update_ball，以及活动中更新、真实试踢和停止后复用。

卷尺提供本次测试的实际球位；静止球与odom只用于隔离控制接口。前3轮数据沿用，不要求先解决视觉定位或从零标定。活动代码受现场人工控制，stop报错时结束后续运动并按既有流程接管。

## STEP 01　连接与记录

仅运行一次。准备和日志函数已放在本格，原样执行，无需阅读或修改辅助函数。SDK方法的实际返回、参数与完整异常会自动保存。模式读取失败不会阻断构造。

重复运行此格会保留已有连接和记录目录。重新开始前先确认机器人控制已停止并人工恢复，再重启内核；内核重启不能代替stop。

In [ ]:
TESTER = "xy"
ROBOT_ID = "060100"

import sys
import json
import time
import math
import traceback
import importlib.metadata
from datetime import datetime
from pathlib import Path
import numpy as np

if "report" in globals() or "robot" in globals():
    raise RuntimeError("已有本次连接或记录，保留原目录；继续后续步骤")
OUTPUT = Path.cwd() / "足球测试结果" / datetime.now().strftime("第4轮完整_%Y%m%d_%H%M%S_%f")
OUTPUT.mkdir(parents=True, exist_ok=False)
report = {"test": "足球SDK完整能力验证_v2", "created_at": datetime.now().isoformat(),
          "tester": TESTER, "robot_id": ROBOT_ID, "calls": [], "cases": {},
          "control_uncertain": False, "active_tests_available": False}

def save():
    (OUTPUT / "原始记录.json").write_text(
        json.dumps(report, ensure_ascii=False, indent=2, default=str, allow_nan=False), encoding="utf-8")

def call(method, function, *args, **kwargs):
    row = {"method": method, "case": report.get("current_case", "准备"),
           "args": list(args), "kwargs": kwargs, "attempted_at": datetime.now().isoformat()}
    report["calls"].append(row)
    save()
    try:
        value = function(*args, **kwargs)
    except (Exception, KeyboardInterrupt) as exc:
        row.update(returned=False, error=repr(exc), traceback=traceback.format_exc())
        save()
        print(method, "报错：", repr(exc))
        raise
    row.update(returned=True, return_type=type(value).__name__, return_repr=repr(value))
    save()
    if method != "get_odom":
        print(method, "返回：", repr(value))
    return value

def modes(label):
    row = {"label": label}
    for name, function in [("public", lambda: robot.get_mode()),
                           ("raw", lambda: robot.loco_client.GetMode().mode)]:
        try:
            row[name] = str(function())
        except Exception as exc:
            row[name + "_error"] = repr(exc)
    report.setdefault("modes", []).append(row)
    save()
    print(label, row)

def pose():
    state = call("get_odom", robot.get_odom)
    values = [float(v) for v in state.pose_2d]
    if len(values) != 3 or not all(math.isfinite(v) for v in values):
        raise ValueError("odom.pose_2d不是有限的[x,y,yaw]")
    report["calls"][-1]["pose_2d"] = values
    save()
    return values

def ball_at_current_frame(ball_xy, measured_pose, current_pose):
    # 仅用于本次教师测试：球保持静止时，利用odom补偿机身平移/转向。
    x, y = ball_xy
    x0, y0, a0 = measured_pose
    x1, y1, a1 = current_pose
    world_x = x0 + math.cos(a0) * x - math.sin(a0) * y
    world_y = y0 + math.sin(a0) * x + math.cos(a0) * y
    dx, dy = world_x - x1, world_y - y1
    return [math.cos(a1) * dx + math.sin(a1) * dy,
            -math.sin(a1) * dx + math.cos(a1) * dy]

def stop_control(case):
    try:
        call("stop", soccer.stop)
    except (Exception, KeyboardInterrupt):
        case["stop_returned"] = False
        report["control_uncertain"] = True
        report["active_tests_available"] = False
        print("stop未返回，请按既有人工控制流程接管；后续运动步骤全部跳过。")
    else:
        case["stop_returned"] = True
        report["control_uncertain"] = False
    save()

def can_run(label):
    if label in report["cases"]:
        print("本项已有记录，不重复运行：", label)
        return False
    if not report.get("manager_created"):
        print("构造未成功，运动部分不可执行；仍可完成末尾汇总。")
        return False
    if report["control_uncertain"]:
        print("控制停止情况待处理，本项跳过；请人工恢复并保存。")
        return False
    return True

def checked_ball(x, y):
    if x is None or y is None:
        print("请填写本次卷尺实测BALL_X、BALL_Y，禁止沿用上一位置的数值。")
        return None
    values = [float(x), float(y)]
    if not all(math.isfinite(v) for v in values) or values[0] <= 0:
        print("球应在前方，x为正且x、y均须为有限米制数。")
        return None
    return values

def baseline():
    label = "无球位启停"
    if not can_run(label):
        return
    if input("确认机器人可进入足球模式，输入“启停”：").strip() != "启停":
        print("本项未启动。")
        return
    row = {"start_returned": False, "stop_returned": False}
    report["cases"][label] = row
    report["current_case"] = label
    try:
        report["control_uncertain"] = True
        call("start", soccer.start)
        row["start_returned"] = True
        time.sleep(1.0)
    except (Exception, KeyboardInterrupt) as exc:
        row["error"] = repr(exc)
    finally:
        stop_control(row)
    report["active_tests_available"] = bool(row["start_returned"] and row["stop_returned"] and not row.get("error"))
    modes("无球位启停后")
    save()

def motion_case(label, x, y, direction=0.0, power=1.0, continuous=False, observe_s=2.0):
    if not can_run(label):
        return
    if not report["active_tests_available"]:
        print("无球位启停尚未正常完成，跳过本项；故障已足以判定当前实验条件不足。")
        return
    xy = checked_ball(x, y)
    if xy is None:
        return
    if continuous and (xy[0] < 1.2 or abs(xy[1]) > 0.3):
        print("连续更新试验请把静止球放在前方至少1.2米、侧向0.3米以内，再实测填写。")
        return
    if not (math.isfinite(observe_s) and 0 < observe_s <= 4.0):
        print("本次观察窗口须在0至4秒内。")
        return
    if input("已重新摆球并实测，现场可试验，输入“执行”：").strip() != "执行":
        print("本项未启动。")
        return
    row = {"measured_ball_xy": xy, "direction": direction, "power": power,
           "continuous": continuous, "samples": [], "start_returned": False,
           "stop_returned": False, "observation_limit_s": observe_s}
    report["cases"][label] = row
    report["current_case"] = label
    save()
    # 测量时机器人和球保持不动，先取位姿，再启动；启动后补偿发生的机身位移。
    try:
        initial_pose = pose()
        row["measured_pose"] = initial_pose
    except (Exception, KeyboardInterrupt) as exc:
        row["error"] = "启动前里程计读取失败：" + repr(exc)
        save()
        return
    try:
        report["control_uncertain"] = True
        call("start", soccer.start)
        row["start_returned"] = True
        call("update_command", soccer.update_command, direction=direction, power=power)
        started = time.monotonic()
        changed = False
        while time.monotonic() - started < observe_s:
            current = pose()
            bx, by = ball_at_current_frame(xy, initial_pose, current)
            displacement = math.hypot(current[0] - initial_pose[0], current[1] - initial_pose[1])
            if continuous and (bx < 0.8 or displacement >= 0.15):
                row["end_reason"] = "达到本次接近试验的行进边界，提前停止"
                break
            call("update_ball", soccer.update_ball, x=bx, y=by)
            row["samples"].append({"odom": current, "ball_xy": [bx, by]})
            save()
            if not continuous:
                # 真实试踢只下发一次当前球位。球被踢动后不再用静止球假设续发。
                time.sleep(observe_s)
                row["end_reason"] = "单次球位输入观察结束；实际是否踢出另记"
                break
            if not changed and len(row["samples"]) >= 2:
                # 活动中实际修改方向和力度；源码允许根据旧球位/odom立即重新发布。
                call("update_command", soccer.update_command, direction=0.20, power=2.0)
                row["active_command_changed"] = True
                changed = True
            time.sleep(0.1)
        row.setdefault("end_reason", "连续更新观察窗口结束")
    except KeyboardInterrupt:
        row["observation_interrupted"] = True
        row["end_reason"] = "人工中断本次观察，finally尝试stop；现场原因另记"
        if not row["start_returned"] or not row["samples"]:
            report["active_tests_available"] = False
    except Exception as exc:
        row["error"] = repr(exc)
        report["active_tests_available"] = False
    finally:
        stop_control(row)
    modes(label + "停止后")
    save()
    print(label, "已记录；球位调用次数：", len(row["samples"]), "；请记录实际表现。")

try:
    report["sdk_version"] = importlib.metadata.version("boosteros")
    from boosteros.robots.booster import BoosterRobot, SoccerKickManager
    robot = call("BoosterRobot", BoosterRobot, enable_tf_listener=True)
except Exception as exc:
    report["connection_error"] = repr(exc)
    save()
    raise
try:
    report["robot_info"] = repr(robot.robot_info)
except Exception as exc:
    report["robot_info_error"] = repr(exc)
modes("连接后")
print("结果目录：", OUTPUT.resolve())

## STEP 02　实际创建、读取里程计，补充TF

原样运行一次。构造独立记录；里程计和TF分别尝试，失败不会被误记为构造失败。TF只是补充信息，后面的手量坐标测试不依赖它。

In [ ]:
if "构造与查询" in report["cases"]:
    raise RuntimeError("本项已有记录，继续STEP 03")
report["current_case"] = "构造与查询"
report["cases"]["构造与查询"] = {}
try:
    soccer = call("SoccerKickManager", SoccerKickManager, robot)
except Exception:
    report["manager_created"] = False
else:
    report["manager_created"] = True
try:
    report["initial_odom"] = pose()
except Exception as exc:
    report["initial_odom_error"] = repr(exc)
try:
    transform = call("get_transform", robot.get_transform,
                     target_frame="trunk", source_frame="head_color_optical_frame")
    matrix = np.asarray(transform.to_matrix())
    if matrix.shape != (4, 4) or not np.isfinite(matrix).all():
        raise ValueError("TF矩阵结构不是有限4×4")
    report["transform"] = {"source": transform.source_frame, "target": transform.target_frame,
                           "matrix": matrix.tolist()}
except Exception as exc:
    report["transform_error"] = repr(exc)
save()
print("管理器实际创建：", report.get("manager_created"))

## STEP 03　无球位启动与停止

先移开足球，确认机器人可进入足球模式，再输入“启停”。start返回后观察一秒，finally中尝试stop。没有球位和指令，不期待踢球。

这两个返回和现场变化分开记录。stop不会自动切回其他模式。若启停失败，仍可执行参数检查与最后汇总，活动试验自动跳过。

In [ ]:
baseline()

## STEP 04　停用时的参数、返回和异常

本格使用停用状态的参数样本，不代表实际球位置。依次调用方向/力度、球位，以及越界力度和非数值球位；预期不合法输入报ValueError。最后调用stop清空全部试验参考，才允许进行活动测试。

这些调用覆盖接口输入和返回；停用状态正常返回不代表已经驱动踢球。

In [ ]:
label = "停用参数检查"
if can_run(label):
    report["cases"][label] = {"checks": []}
    report["current_case"] = label
    row = report["cases"][label]
    try:
        call("update_command", soccer.update_command, direction=0.0, power=1.0)
        call("update_command", soccer.update_command, direction=-0.25, power=10.0)
        call("update_ball", soccer.update_ball, x=0.6, y=0.0)
        invalid = [("power=0", soccer.update_command, {"direction": 0.0, "power": 0.0}),
                   ("power=11", soccer.update_command, {"direction": 0.0, "power": 11.0}),
                   ("x非数值", soccer.update_ball, {"x": "invalid", "y": 0.0})]
        for name, function, kwargs in invalid:
            method = "update_ball" if name == "x非数值" else "update_command"
            try:
                call(method, function, **kwargs)
            except ValueError:
                row["checks"].append({"input": name, "expected_ValueError": True})
            except Exception as exc:
                row["checks"].append({"input": name, "unexpected_error": repr(exc)})
            else:
                row["checks"].append({"input": name, "expected_ValueError": False})
    except (Exception, KeyboardInterrupt) as exc:
        row["error"] = repr(exc)
    finally:
        stop_control(row)
    save()
    print("停用参数检查已记录；未把参数样本作为真实球位试踢。")

## STEP 05　远处静止球：连续球位与活动中指令更新

将球放在前方至少1.2米、侧向0.3米以内，机器人和球保持不动，实测后填写本格两个数值。执行后球保持静止。代码补偿启动后的机身位移，连续发送当前机身坐标；两次球位后再把方向改为0.20 rad、力度改为2。

观察窗口设为2秒，读到行进达到0.15米或估计球距小于0.8米会提前stop。若球被碰动，立即中断当前格。此步验证持续输入与活动中更新，不判定实际踢球力度；临时odom换算不属于学生必修代码。

In [ ]:
BALL_X = None  # 填本项实测前向距离（米），至少1.2
BALL_Y = None  # 填本项实测侧向距离（米），左正右负
motion_case("连续球位与活动中指令更新", BALL_X, BALL_Y, continuous=True, observe_s=2.0)

## STEP 06　基线真实试踢

恢复机器人、重新摆球。按实际坐标原点量好本次球位，填写本格数值，再输入“执行”。方向为正前方、力度为1。只下发一次启动后补偿的当前球位，再等待2秒观察；看到球踢动可立即中断。

记录是否接近、是否踢动、实际方向以及stop后的表现。未踢动也保留结果，不自行延长到无人看护的持续运行。

In [ ]:
BALL_X = None  # 重新实测；约0.6米只是官方示例，不是直接可用的固定值
BALL_Y = None
motion_case("基线试踢", BALL_X, BALL_Y, direction=0.0, power=1.0, observe_s=2.0)

## STEP 07　改变方向，停止后复用

重新恢复并实测球位，复用原管理器重新start。本项方向改为0.25 rad（约向左14°）、力度仍为1。记录重启是否正常、方向作用是否可观察。结束仍stop。

即使前一项未踢动，只要接口无异常、现场条件正常，也可继续本项；如有异常或stop失败，代码自动跳过后续活动。

In [ ]:
BALL_X = None  # 本项重新测量
BALL_Y = None
motion_case("改方向与重启", BALL_X, BALL_Y, direction=0.25, power=1.0, observe_s=2.0)

## STEP 08　改变力度，再次复用

重新恢复并实测球位。本项方向回到0，力度改为2。与STEP 06比较是否有可辨别变化；地面、摆球和单次输入都会影响实际表现，没有明显区别时记录“本次无法区分”。不作精度、力度比例或成功率测试。

In [ ]:
BALL_X = None  # 本项重新测量
BALL_Y = None
motion_case("改力度与重启", BALL_X, BALL_Y, direction=0.0, power=2.0, observe_s=2.0)

## STEP 09　保存覆盖清单与课程决策记录

原样运行。本格保存当前SDK管理器和TF适配层源码、自动方法覆盖表。表中“正常返回”是调用事实，不等于踢球成功；参数检查中的预期ValueError另见原始JSON。被跳过的项目不会被记成已验证。

填写[现场记录](足球测试_第4轮_操作与记录.md)，把整个结果目录复制回知识库后告诉助手“完整测试已记录”。助手据此给出U7/U8课程删改结论，不默认再拆下一轮。stop报错时先按已有人工流程恢复机器人。

In [ ]:
import inspect
sources = [("SoccerKickManager", SoccerKickManager)] if "SoccerKickManager" in globals() else []
if "robot" in globals():
    sources += [("BoosterRobot_get_mode", type(robot).get_mode)]
    try:
        sources.append(("RosAdapter_get_transform", robot.ros_adapter.get_transform))
    except Exception:
        pass
report["sdk_sources"] = []
for name, obj in sources:
    row = {"name": name}
    try:
        filename = name + "_本机实现.txt"
        (OUTPUT / filename).write_text(inspect.getsource(obj), encoding="utf-8")
        row["source_file"] = filename
    except Exception as exc:
        row["error"] = repr(exc)
    report["sdk_sources"].append(row)
save()
lines = ["# 足球SDK完整能力测试汇总", "", "时间：" + report["created_at"],
         "SDK：" + report.get("sdk_version", "未取得"), "",
         "| 方法 | 调用次数 | 正常返回次数 | 返回类型/值 |", "| --- | --- | --- | --- |"]
for method in ["SoccerKickManager", "start", "stop", "update_command", "update_ball"]:
    rows = [r for r in report["calls"] if r["method"] == method]
    returned = [r for r in rows if r.get("returned")]
    values = sorted(set(r["return_type"] + ":" + r["return_repr"] for r in returned))
    line = "| " + method + " | " + str(len(rows)) + " | " + str(len(returned)) + " | " + "; ".join(values) + " |"
    lines.append(line)
    print(line)
lines += ["", "## 场景", "", "| 项目 | 代码执行结果 |", "| --- | --- |"]
for label in ["构造与查询", "无球位启停", "停用参数检查", "连续球位与活动中指令更新", "基线试踢", "改方向与重启", "改力度与重启"]:
    row = report["cases"].get(label)
    status = "未执行" if row is None else json.dumps(row, ensure_ascii=False, default=str)
    lines.append("| " + label + " | " + status.replace("|", "/") + " |")
lines += ["", "现场球是否移动、方向/力度效果、stop后表现与测量原点见人工记录。",
          "SDK直接视觉米制球位来源尚未证实；手量+静止球odom测试不能替代视觉定位。",
          "回读后作U7/U8课程决策；未执行项和故障不能写成已验证。"]
(OUTPUT / "能力汇总.md").write_text("\n".join(lines), encoding="utf-8")
print("结果目录：", OUTPUT.resolve())
print("填写现场记录后交助手，进入课程边界与实验设计修订。")